In [1]:
import pandas as pd
import numpy as np
import transformers
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from transformers import AutoTokenizer
import torch
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import torch.nn as nn
from sklearn.metrics import accuracy_score
import pickle as pkl
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import string

/home/i666sapple/.local/lib/python3.12/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
class Config:
    CLS = [101]
    SEP = [102]
    VALUE_TOKEN = [0]
    MAX_LEN = 128
    TRAIN_BATCH_SIZE = 32
    VAL_BATCH_SIZE = 8
    EPOCHS = 7
    TOKENIZER = AutoTokenizer.from_pretrained("sagorsarker/mbert-bengali-ner")

In [3]:
class Dataset:
    def __init__(self, texts, tags):
        self.texts = texts
        self.tags = tags
  
    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        texts = self.texts[index]
        tags = self.tags[index]
    
        # Tokenize
        ids = []
        target_tag = []

        for i, s in enumerate(texts):
            inputs = Config.TOKENIZER.encode(s, add_special_tokens=False)
            input_len = len(inputs)
            ids.extend(inputs)
            target_tag.extend(input_len * [tags[i]])
        
        # Add Special Tokens, subtract 2 from MAX_LEN
        ids = ids[:Config.MAX_LEN - 2]
        target_tag = target_tag[:Config.MAX_LEN - 2]

        # Add Special Tokens
        ids = Config.CLS + ids + Config.SEP
        target_tag = Config.VALUE_TOKEN + target_tag + Config.VALUE_TOKEN

        mask = [1] * len(ids)
        token_type_ids = [0] * len(ids)

        # Add Padding if the input_len is small
        padding_len = Config.MAX_LEN - len(ids)
        ids = ids + ([0] * padding_len)
        target_tag = target_tag + ([0] * padding_len)
        mask = mask + ([0] * padding_len)
        token_type_ids = token_type_ids + ([0] * padding_len)

        return {
            "ids": torch.tensor(ids, dtype=torch.long),
            "mask": torch.tensor(mask, dtype=torch.long),
            "token_type_ids": torch.tensor(token_type_ids, dtype=torch.long),
            "target_tags": torch.tensor(target_tag, dtype=torch.long)
        }

In [4]:
def train_fn(train_data_loader, model, optimizer, device, scheduler):
    model.train()
    loss_ = 0
    for data in tqdm(train_data_loader, total=len(train_data_loader)):
        for i, j in data.items():
            data[i] = j.to(device)

        optimizer.zero_grad()
        _, loss = model(**data)
        loss.backward()
        optimizer.step()
        scheduler.step()
        loss_ += loss.item()
    return model, loss_ / len(train_data_loader)

In [5]:
def val_fn(val_data_loader, model, optimizer, device, scheduler):
    model.eval()
    loss_ = 0
    for data in tqdm(val_data_loader, total=len(val_data_loader)):
        for i, j in data.items():
            data[i] = j.to(device)
        _, loss = model(**data)
        loss_ += loss.item()
    return loss_ / len(val_data_loader)

In [6]:
from transformers import AutoModel

class NERBertModel(nn.Module):
    def __init__(self, num_tag):
        super(NERBertModel, self).__init__()
        self.num_tag = num_tag
        self.bert = AutoModel.from_pretrained("sagorsarker/mbert-bengali-ner")
        self.bert_drop = nn.Dropout(0.3)
        self.out_tag = nn.Linear(768, self.num_tag)
        
    def forward(self, ids, mask, token_type_ids, target_tags):
        output = self.bert(ids, attention_mask=mask, token_type_ids=token_type_ids)
        bert_out = self.bert_drop(output.last_hidden_state)
        tag = self.out_tag(bert_out)
    
        # Calculate the loss
        criterion_loss = nn.CrossEntropyLoss()
        active_loss = mask.view(-1) == 1
        active_logits = tag.view(-1, self.num_tag)
        active_labels = torch.where(active_loss, target_tags.view(-1), torch.tensor(criterion_loss.ignore_index).type_as(target_tags))
        loss = criterion_loss(active_logits, active_labels)
        return tag, loss

In [7]:
from sklearn.preprocessing import LabelEncoder

def parse_dataset(file_path):
    sentences = []
    words = []
    pos_tags = []
    ner_tags = []
    malformed_lines = []

    with open(file_path, 'r', encoding='utf-8') as file:
        current_sentence = []
        current_pos = []
        current_ner = []

        for idx, line in enumerate(file):
            line = line.strip()
            if line:
                if '\t' in line:
                    parts = line.split('\t')
                    if len(parts) == 3:
                        token, pos, ner = parts
                        current_sentence.append(token)
                        current_pos.append(pos)
                        current_ner.append(ner)
                    else:
                        malformed_lines.append((idx, line))
                else:
                    if current_sentence:
                        sentences.append(" ".join(current_sentence))
                        words.append(current_sentence)
                        pos_tags.append(current_pos)
                        ner_tags.append(current_ner)
                        current_sentence = []
                        current_pos = []
                        current_ner = []
            else:
                if current_sentence:
                    sentences.append(" ".join(current_sentence))
                    words.append(current_sentence)
                    pos_tags.append(current_pos)
                    ner_tags.append(current_ner)
                    current_sentence = []
                    current_pos = []
                    current_ner = []

        if current_sentence:  # To handle the last sentence in the file
            sentences.append(" ".join(current_sentence))
            words.append(current_sentence)
            pos_tags.append(current_pos)
            ner_tags.append(current_ner)

    df = pd.DataFrame({
        'sentence': sentences,
        'words': words,
        'pos': pos_tags,
        'ner': ner_tags
    })

    pos_set = set(tag for sublist in df['pos'] for tag in sublist)
    ner_set = set(tag for sublist in df['ner'] for tag in sublist)

    pos_mapping = {tag: idx for idx, tag in enumerate(sorted(pos_set))}
    ner_mapping = {tag: idx for idx, tag in enumerate(sorted(ner_set))}

    df['postagid'] = df['pos'].apply(lambda tags: [pos_mapping[tag] for tag in tags])
    df['nertagid'] = df['ner'].apply(lambda tags: [ner_mapping[tag] if isinstance(tag, str) else tag for tag in tags])

    def split_ner_tags(tags):
        before_dash = []
        after_dash = []
        for tag in tags:
            if isinstance(tag, str) and '-' in tag:
                before, after = tag.split('-', 1)
                before_dash.append(before)
                after_dash.append(after)
            else:
                before_dash.append(tag)
                after_dash.append('')
        return before_dash, after_dash

    df['ner_str'] = df['ner'].apply(lambda tags: [tag for tag in tags])
    df[['ner_tag_general', 'ner_tag_pos']] = df['ner_str'].apply(lambda tags: pd.Series(split_ner_tags(tags)))
    df.drop('ner_str', axis=1, inplace=True)

    ner_tag_general_set = set(df['ner_tag_general'].explode().unique())
    ner_tag_pos_set = set(df['ner_tag_pos'].explode().unique())

    ner_tag_general_mapping = {tag: idx for idx, tag in enumerate(sorted(ner_tag_general_set))}
    ner_tag_pos_mapping = {tag: idx for idx, tag in enumerate(sorted(ner_tag_pos_set))}

    df['ner_tag_general'] = df['ner_tag_general'].apply(lambda tags: [ner_tag_general_mapping.get(tag, -1) for tag in tags])
    df['ner_tag_pos'] = df['ner_tag_pos'].apply(lambda tags: [ner_tag_pos_mapping.get(tag, -1) for tag in tags])

    # Fit the LabelEncoder with NER tags
    le = LabelEncoder()
    all_ner_tags = [tag for sublist in df['ner'] for tag in sublist]
    le.fit(all_ner_tags)

    return df, le, malformed_lines

file_path = 'Data/data.tsv'
df, le, malformed_lines = parse_dataset(file_path)

In [8]:
df = df.drop(columns=['sentence', 'ner_tag_general', 'ner_tag_pos'])

In [9]:
df

,words,pos,ner,postagid,nertagid
0,"[শনিবার, (২৭, আগস্ট), রাতে, পটুয়াখালী, সদর, থ...","[NNP, PUNCT, NNP, NNC, NNP, NNC, NNC, ADJ, NNC...","[B-D&T, B-OTH, B-D&T, B-D&T, B-GPE, I-GPE, I-G...","[6, 11, 6, 5, 6, 5, 5, 0, 5, 11, 6, 6, 3, 5, 0...","[0, 7, 0, 0, 2, 13, 13, 8, 18, 7, 8, 18, 7, 7,..."
1,"[বায়ুদূষণ, ও, স্মার্ট, ফোন, ছেলেমেয়ে, উভয়ের, প...","[NNC, CONJ, NNC, NNC, NNC, PRO, NNC, NNC, NNC,...","[B-OTH, B-OTH, B-OTH, B-OTH, B-PER, B-OTH, B-O...","[5, 2, 5, 5, 5, 10, 5, 5, 5, 14, 13]","[7, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7]"
2,"[ছাত্র, রাজনীতির, বর্তমান, অবস্থার, শুরু, হয়ে...","[NNC, NNC, ADJ, NNC, NNC, VF, NNC, NNP, NNC, PP]","[B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-P...","[5, 5, 0, 5, 5, 13, 5, 6, 5, 9]","[7, 7, 7, 7, 7, 7, 8, 8, 7, 7]"
3,"[শাকিল, রাজধানীর, ৩০০, ফিট,, দিয়াবাড়ি, ও, পূ...","[NNP, NNC, QF, NNC, NNP, CONJ, NNP, NNC, NNC, ...","[B-PER, B-OTH, B-LOC, I-LOC, B-LOC, B-OTH, B-L...","[6, 5, 12, 5, 6, 2, 6, 5, 5, 9, 5, 14, 13, 9, ...","[8, 7, 3, 14, 3, 7, 3, 7, 7, 7, 8, 7, 7, 7, 7, 6]"
4,"[সম্প্রতি, ক্লাবের, নবীন, ব্যবস্থাপনা, প্রশিক্...","[ADV, NNC, ADJ, NNC, NNC, CONJ, NNC, NNC, PP, ...","[B-OTH, B-ORG, B-OTH, B-OTH, B-PER, B-OTH, B-P...","[1, 5, 0, 5, 5, 2, 5, 5, 9, 0, 13, 5, 5]","[7, 6, 7, 7, 8, 7, 8, 8, 7, 7, 7, 1, 12]"
...,...,...,...,...,...
6997,"[বিপদ:, প্লাস্টিকের, ব্যাগগুলি, বিপজ্জনক, হতে,...","[NNC, NNC, NNC, ADJ, VNF, VF]","[B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-OTH]","[5, 5, 5, 0, 14, 13]","[7, 7, 7, 7, 7, 7]"
6998,"[তবে, ভুয়া, বিজ্ঞাপন, বন্ধে, ফেসবুকে, সঙ্গে, ...","[CONJ, ADJ, NNC, NNC, NNP, PP, NNC, VF, NNP]","[B-OTH, B-OTH, B-OTH, B-OTH, B-ORG, B-OTH, B-O...","[2, 0, 5, 5, 6, 9, 5, 13, 6]","[7, 7, 7, 7, 6, 7, 7, 7, 8]"
6999,"[১., ২০১৮, সালে, ঘোষিত, সরকারি, চাকরিতে, কোটা,...","[OTH, QF, NNC, ADJ, ADJ, NNC, NNC, NNC, ADJ, C...","[B-NUM, B-D&T, I-D&T, B-OTH, B-OTH, B-OTH, B-O...","[7, 12, 5, 0, 0, 5, 5, 5, 0, 2, 0, 5, 5, 5, 14...","[5, 0, 11, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7]"
7000,"['কয়েদীরা, দিনপঞ্জীর, মতো, দেয়ালে, নিজেদের, স্...","[PUNCT, NNC, PP, NNC, PRO, NNC, VNF, VF]","[B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-OTH, B-O...","[11, 5, 9, 5, 10, 5, 14, 13]","[7, 7, 7, 7, 7, 7, 7, 7]"


In [10]:
# df = df.drop(columns=['sentence', 'ner_tag_general', 'ner_tag_pos'])

In [11]:
df['nertagid'] = df['nertagid'].apply(lambda tags: [tag if not pd.isna(tag) else 0 for tag in tags])

In [12]:
train_sent, val_sent, train_tag, val_tag = train_test_split(df['words'], df['nertagid'], test_size=0.25, random_state=42)

In [13]:
train_dataset = Dataset(texts=train_sent.tolist(), tags=train_tag.tolist())
val_dataset = Dataset(texts=val_sent.tolist(), tags=val_tag.tolist())
train_data_loader = DataLoader(train_dataset, batch_size=Config.TRAIN_BATCH_SIZE)
val_data_loader = DataLoader(val_dataset, batch_size=Config.VAL_BATCH_SIZE)


In [14]:
for data_ in train_data_loader:
    print(data_)
    break

{'ids': tensor([[  101,   608, 23006,  ...,     0,     0,     0],
        [  101,   614, 29425,  ...,     0,     0,     0],
        [  101,   665, 14464,  ...,     0,     0,     0],
        ...,
        [  101, 65235, 75509,  ...,     0,     0,     0],
        [  101,   648, 91252,  ...,     0,     0,     0],
        [  101, 14565,   625,  ...,     0,     0,     0]]), 'mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'target_tags': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 7, 7,  ..., 0, 0, 0],
        [0, 5, 5,  ..., 0, 0, 0],
        ...,
        [0, 6, 6,  ..., 0, 0, 0],
 

In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

num_tag = len(df.nertagid.value_counts())
model = NERBertModel(num_tag=num_tag)
model.to(device)
print(model)

/home/i666sapple/.local/lib/python3.12/site-packages/torch/cuda/__init__.py:118: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0
Some weights of BertModel were not initialized from the model checkpoint at sagorsarker/mbert-bengali-ner and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


NERBertModel(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(105879, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise

In [16]:
def get_hyperparameters(model, ff):
    # ff: full_finetuning
    if ff:
        param_optimizer = list(model.named_parameters())
        no_decay = ["bias", "gamma", "beta"]
        optimizer_grouped_parameters = [
            {
                "params": [
                    p for n, p in param_optimizer if not any(nd in n for nd in no_decay)
                ],
                "weight_decay_rate": 0.01,
            },
            {
                "params": [
                    p for n, p in param_optimizer if any(nd in n for nd in no_decay)
                ],
                "weight_decay_rate": 0.0,
            },
        ]
    else:
        param_optimizer = list(model.classifier.named_parameters())
        optimizer_grouped_parameters = [{"params": [p for n, p in param_optimizer]}]
    return optimizer_grouped_parameters

In [17]:
FULL_FINETUNING = True
optimizer_grouped_parameters = get_hyperparameters(model, FULL_FINETUNING)
optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=3e-5)
num_train_steps = int(len(train_sent) / Config.TRAIN_BATCH_SIZE * Config.EPOCHS)
scheduler = get_linear_schedule_with_warmup(
    optimizer, 
    num_warmup_steps=0, 
    num_training_steps=num_train_steps
)

In [18]:
for epoch in range(Config.EPOCHS):
    model, train_loss = train_fn(train_data_loader, model, optimizer, device, scheduler)
    val_loss = val_fn(val_data_loader, model, optimizer, device, scheduler)
    print(f"Epoch: {epoch + 1}, Train_loss: {train_loss}, Val_loss: {val_loss}")

100%|██████████| 219/219 [03:06<00:00,  1.18it/s]


Epoch: 1, Train_loss: 2.076490790193731, Val_loss: 0.78300457512407


100%|██████████| 219/219 [03:14<00:00,  1.13it/s]


Epoch: 2, Train_loss: 0.6274560234311856, Val_loss: 0.45388529929396226


100%|██████████| 219/219 [03:05<00:00,  1.18it/s]


...

100%|██████████| 219/219 [03:15<00:00,  1.12it/s]


Epoch: 5, Train_loss: 0.20686980107742728, Val_loss: 0.358449263719323


100%|██████████| 219/219 [03:11<00:00,  1.14it/s]


Epoch: 6, Train_loss: 0.1686996604682821, Val_loss: 0.35054471729893116


 21%|██        | 34/165 [06:32<26:54, 12.33s/it]

In [ ]:
def prediction(test_sentence, model, tokenizer, le):
    # Replace punctuation with space-punctuation
    for i in list(string.punctuation):
        test_sentence = test_sentence.replace(i, ' ' + i)
    
    # Tokenize the test sentence
    test_sentence_tokens = test_sentence.split()
    
    # Encode the test sentence with the tokenizer
    encoded_inputs = tokenizer(test_sentence_tokens, is_split_into_words=True, return_tensors="pt", padding=True, truncation=True)
    
    # Prepare the test dataset
    input_ids = encoded_inputs["input_ids"].to(device)
    attention_mask = encoded_inputs["attention_mask"].to(device)
    token_type_ids = encoded_inputs.get("token_type_ids", torch.zeros_like(input_ids)).to(device)
    
    # Placeholder target tags
    target_tags = torch.zeros(input_ids.shape, dtype=torch.long).to(device)
    
    with torch.no_grad():
        # Get the model predictions
        outputs = model(input_ids, attention_mask, token_type_ids, target_tags)
        tag = outputs[0]  # Assuming the first element is the tag predictions
        predicted_indices = tag.argmax(2).cpu().numpy().reshape(-1)
        
        # Convert indices to NER tags
        predicted_tags = le.inverse_transform(predicted_indices)[1:len(test_sentence_tokens)+1]
        
        # Print the result
        result = list(zip(test_sentence_tokens, predicted_tags))
        print(result)
        return result

NotFittedError: This LabelEncoder instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.

In [ ]:
# Load your model and set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

In [ ]:
# Test the prediction function
test_sentence = "বাংলাদেশের ইতিহাসে বাংলাদেশ কখনওই রাষ্ট্র হিসেবে একটি সফল হতে পারেনি"
result = prediction(test_sentence, model, Config.TOKENIZER, le)
print(result)

TypeError: NERBertModel.forward() missing 2 required positional arguments: 'token_type_ids' and 'target_tags'

In [ ]:
torch.save(model.state_dict(), "ner_model.pt")

[('বাংলাদেশের', 'B-GPE'), ('ইতিহাসে', 'B-OTH'), ('বাংলাদেশ', 'B-OTH'), ('কখনওই', 'B-OTH'), ('রাষ্ট্র', 'B-OTH'), ('হিসেবে', 'B-OTH'), ('একটি', 'B-GPE'), ('সফল', 'B-OTH'), ('হতে', 'B-OTH'), ('পারেনি', 'B-OTH')]
[('বাংলাদেশের', 'B-GPE'), ('ইতিহাসে', 'B-OTH'), ('বাংলাদেশ', 'B-OTH'), ('কখনওই', 'B-OTH'), ('রাষ্ট্র', 'B-OTH'), ('হিসেবে', 'B-OTH'), ('একটি', 'B-GPE'), ('সফল', 'B-OTH'), ('হতে', 'B-OTH'), ('পারেনি', 'B-OTH')]
